<a href="https://colab.research.google.com/github/boboren2/Team-14/blob/main/Maincode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd

BASE = "/content/drive/MyDrive/Project Labs/"
FILES = ["pp-2023.csv", "pp-2024.csv", "pp-2025 (1).csv"]

COLS = ["id","price","date","postcode","prop_type","new_build","tenure",
        "paon","saon","street","locality","town","district","county",
        "ppd_cat","record_status"]
KEEP = ["id","price","date","postcode","prop_type","new_build","tenure",
        "district","ppd_cat","record_status"]
AREAS = ["EAST RIDING OF YORKSHIRE", "EAST LINDSEY"]

parts = []
for f in FILES:
    df = pd.read_csv(BASE + f, header=None, names=COLS, usecols=KEEP,
                     keep_default_na=False)
    print(f, "rows in whole country file:", len(df))
    parts.append(df[df["district"].isin(AREAS)])

sales = pd.concat(parts, ignore_index=True)
print("Rows in the two authorities:", len(sales))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
pp-2023.csv rows in whole country file: 861782
pp-2024.csv rows in whole country file: 932214
pp-2025 (1).csv rows in whole country file: 992278
Rows in the two authorities: 29916


In [2]:
(sales == "").sum()

,0
id,0
price,0
date,0
postcode,51
prop_type,0
new_build,0
tenure,0
district,0
ppd_cat,0
record_status,0


In [3]:
print("Repeated ids:", sales["id"].duplicated().sum())
sales = sales.drop_duplicates("id")

Repeated ids: 0


In [4]:
#irregular spellings
for col in ["district", "prop_type", "new_build", "tenure", "ppd_cat", "record_status"]:
    print(sales[col].value_counts(), "\n")

sales["postcode"] = sales["postcode"].str.strip().str.upper()
print("Postcodes in wrong format:",
      (~sales["postcode"].str.match(r"^[A-Z]{1,2}\d[A-Z\d]? \d[A-Z]{2}$")).sum())


district
EAST RIDING OF YORKSHIRE    21861
EAST LINDSEY                 8055
Name: count, dtype: int64 

prop_type
D    11259
S     9725
T     5747
O     1671
F     1514
Name: count, dtype: int64 

new_build
N    24915
Y     5001
Name: count, dtype: int64 

tenure
F    28031
L     1885
Name: count, dtype: int64 

ppd_cat
A    25139
B     4777
Name: count, dtype: int64 

record_status
A    29916
Name: count, dtype: int64 

Postcodes in wrong format: 51


In [5]:
sales["date"] = pd.to_datetime(sales["date"], format="%Y-%m-%d %H:%M")
sales["price"] = pd.to_numeric(sales["price"])
print(sales[["price", "date"]].describe())

              price                           date
count  2.991600e+04                          29916
mean   2.578691e+05  2024-07-27 04:24:30.036101120
min    1.000000e+02            2023-01-03 00:00:00
25%    1.567490e+05            2023-10-26 00:00:00
50%    2.150000e+05            2024-08-14 00:00:00
75%    3.000000e+05            2025-04-17 00:00:00
max    2.910000e+07            2025-12-23 00:00:00
std    3.424781e+05                            NaN


In [6]:
print("Not standard sale (ppd_cat B):", (sales["ppd_cat"] != "A").sum())
print("Property type Other:", (sales["prop_type"] == "O").sum())
print("No postcode:", (sales["postcode"] == "").sum())
print("Tenure not F or L:", (~sales["tenure"].isin(["F", "L"])).sum())
print("Price of 0 or less:", (sales["price"] <= 0).sum())
print("Dates outside 2023-2025:", (~sales["date"].dt.year.between(2023, 2025)).sum())

sales = sales[
    (sales["ppd_cat"] == "A")
    & (sales["prop_type"] != "O")
    & (sales["postcode"] != "")
    & (sales["tenure"].isin(["F", "L"]))
    & (sales["price"] > 0)
].copy()

Not standard sale (ppd_cat B): 4777
Property type Other: 1671
No postcode: 51
Tenure not F or L: 0
Price of 0 or less: 0
Dates outside 2023-2025: 0


In [7]:
sales["year"] = sales["date"].dt.year
sales["pc_district"] = sales["postcode"].str.split(" ").str[0]
sales = sales[["id","price","date","year","postcode","pc_district",
               "prop_type","new_build","tenure","district"]]

print(sales.shape)
print(sales.groupby(["district", "year"]).size().unstack())
sales.to_csv(BASE + "sales_clean.csv", index=False)
print("Saved", len(sales), "rows")

(25129, 10)
year                      2023  2024  2025
district                                  
EAST LINDSEY              2061  2167  2403
EAST RIDING OF YORKSHIRE  5746  6066  6686
Saved 25129 rows
